In [3]:
# ============================================================
# Stage 5 — Compare with Federico Malizia et al.
# Cell 0: setup and global configuration
#
# Goal of this notebook:
#   1. Reproduce the Malizia Rössler dynamical system in Python
#   2. Verify Python dynamics against the original MATLAB output
#   3. Convert the system to the TSC endpoint-data protocol
#   4. Level 1: infer using the oracle interaction functions
#   5. Level 2: infer using only interaction-order / polynomial-degree priors
#
# Important:
#   This notebook does NOT reimplement Signal Lasso.
#   MATLAB reproduction has already been used to validate the
#   original Malizia inference pipeline.
# ============================================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp
from scipy.io import loadmat

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 1
rng = np.random.default_rng(SEED)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "stage5_fm_outputs"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# Original Malizia / MATLAB files
#
# Put these either:
#   - in the notebook directory, or
#   - in ./data/
#
# ZackaryNet.mat:
#   original Karate Club network / triangle information
#
# Malizia_iteration_sensitivity_MH01.mat:
#   file saved from our MATLAB iteration-sensitivity experiment;
#   contains x0, T, X, Phi_all, Y_all, AA, etc.

def locate_file(filename):
    candidates = [
        ROOT / filename,
        DATA_DIR / filename,
    ]

    for path in candidates:
        if path.exists():
            return path

    return None


ZACKARY_FILE = locate_file("./dataset/ZackaryNet.mat")

MATLAB_CHECK_FILE = locate_file(
    "./dataset/Malizia_iteration_sensitivity_MH01.mat"
)

# ------------------------------------------------------------
# Malizia Rössler benchmark parameters
# ------------------------------------------------------------

N_EXPECTED = 34

# Pairwise coupling strength
K_PAIR = 1e-4

# Three-body coupling strength
K_TRI = 1e-5

# Original integration horizon
TMAX_ORIGINAL = 100.0

# Tight tolerance used for MATLAB/Python equivalence checks
RTOL_CHECK = 1e-12
ATOL_CHECK = 1e-12

# scipy RK45 is the closest direct analogue here to MATLAB ode45
IVP_METHOD = "RK45"

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

np.set_printoptions(
    precision=6,
    suppress=True,
    linewidth=140
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

# ------------------------------------------------------------
# Environment summary
# ------------------------------------------------------------

print("=" * 70)
print("Stage5_Compare_FM")
print("=" * 70)

print(f"Python version : {sys.version.split()[0]}")
print(f"NumPy version  : {np.__version__}")
print(f"Random seed    : {SEED}")

print()
print("Paths")
print("-" * 70)
print(f"ROOT           : {ROOT}")
print(f"DATA_DIR       : {DATA_DIR}")
print(f"OUTPUT_DIR     : {OUTPUT_DIR}")

print()
print("Input files")
print("-" * 70)
print(f"ZackaryNet.mat : {ZACKARY_FILE}")
print(f"MATLAB check   : {MATLAB_CHECK_FILE}")

print()
print("Malizia benchmark")
print("-" * 70)
print(f"Expected N     : {N_EXPECTED}")
print(f"K_PAIR         : {K_PAIR:.1e}")
print(f"K_TRI          : {K_TRI:.1e}")
print(f"IVP method     : {IVP_METHOD}")
print(f"RTOL / ATOL    : {RTOL_CHECK:.1e} / {ATOL_CHECK:.1e}")

print("=" * 70)

Stage5_Compare_FM
Python version : 3.13.5
NumPy version  : 2.1.3
Random seed    : 1

Paths
----------------------------------------------------------------------
ROOT           : C:\Users\liu.xuanc\Desktop\Code\TSC-TemporalStructureClosure
DATA_DIR       : C:\Users\liu.xuanc\Desktop\Code\TSC-TemporalStructureClosure\data
OUTPUT_DIR     : C:\Users\liu.xuanc\Desktop\Code\TSC-TemporalStructureClosure\stage5_fm_outputs

Input files
----------------------------------------------------------------------
ZackaryNet.mat : C:\Users\liu.xuanc\Desktop\Code\TSC-TemporalStructureClosure\dataset\ZackaryNet.mat
MATLAB check   : C:\Users\liu.xuanc\Desktop\Code\TSC-TemporalStructureClosure\dataset\Malizia_iteration_sensitivity_MH01.mat

Malizia benchmark
----------------------------------------------------------------------
Expected N     : 34
K_PAIR         : 1.0e-04
K_TRI          : 1.0e-05
IVP method     : RK45
RTOL / ATOL    : 1.0e-12 / 1.0e-12
